# Analiza streszczeń literatury młodzieżowej

# Import bibliotek

In [ ]:
import os
import json
import nltk
import string
import morfeusz2
import pandas as pd
import operator as op
import itertools as it
from wordcloud import WordCloud
from matplotlib import pyplot as plt
from nltk.tokenize import word_tokenize
from nltk.corpus import PlaintextCorpusReader
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

In [ ]:
nltk.download('punkt_tab')
nltk.pathsec.ENFORCE = False

# Zdefiniowanie funckji pomocnicznych

In [ ]:
def print_dict(obj):
    print(json.dumps(obj, indent=4, ensure_ascii=False))
    

In [ ]:
def create_dir(path):
    if not os.path.exists():
        os.mkdir(path)

# Utworzenie korpusu dokumentów

In [ ]:
corpus_dir = "./Literature"
corpus = PlaintextCorpusReader(corpus_dir, r".*\.txt")
files_names = corpus.fileids()

In [ ]:
documents_dict = {}
for file_name in files_names:
    documents_dict[file_name] = corpus.raw(file_name)
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
lengths = {}
for file_name in documents_dict:
    lengths[file_name] = {
        "pre": len(word_tokenize(documents_dict[file_name])),
    }
print_dict(lengths)

## Utworzenie macierzy częstości

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
matrix_tf = CountVectorizer().fit_transform(documents_df_post['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices = {
    "pre": {
        "words": matrix_tf.shape[1],
        "sparsity": sparsity_tf
    }
}

In [ ]:
matrices = pd.DataFrame.from_dict(matrices, orient="index")
display(matrices)

# Wstępne przygotowanie dokumentów

In [ ]:
stopwords_file = open("./stopwords_pl.txt", "r", encoding="utf-8")
stopwords = stopwords_file.read().splitlines()
stopwords = stopwords[4:]  # Skip the first four lines of the file
stopwords_file.close()
stopwords

In [ ]:
def lemmatize(text):
    morfer = morfeusz2.Morfeusz()
    segments = it.groupby(morfer.analyse(text), op.itemgetter(0,1))
    def disambiguate(group):
        pairs = ((len(descr), lemma) for _, _, (_, _, lemma, descr, _, _, ) in group)
        perpl, lemma = min(pairs)
        return lemma.split(':')[0] if lemma else None
    lemmas = (disambiguate(group) for _, group in segments)
    return ' '.join(filter(str.isalpha, lemmas))

In [ ]:
for file_name in documents_dict:
    documents_dict[file_name] = documents_dict[file_name].lower()
    documents_dict[file_name] = "".join([char for char in documents_dict[file_name] if char not in string.punctuation])
    documents_dict[file_name] = lemmatize(documents_dict[file_name])
    documents_dict[file_name] = " ".join([word for word in word_tokenize(documents_dict[file_name], language='polish') if word not in stopwords])
print(json.dumps(documents_dict, indent=4, ensure_ascii=False))

In [ ]:
for file_name in documents_dict:
    lengths[file_name]["post"] = len(word_tokenize(documents_dict[file_name]))
print(json.dumps(lengths, indent=4, ensure_ascii=False))

In [ ]:
lengths = pd.DataFrame.from_dict(lengths, orient='index')
lengths['change'] = lengths['post'] - lengths['pre']
lengths['change_percentage'] = ((lengths['change'] / lengths['pre']) * 100).round(2)
lengths

# Utowrzenie macierzy częstości

In [ ]:
documents_df_post = pd.DataFrame.from_dict(documents_dict, orient='index', columns=['content'])

In [ ]:
cv = CountVectorizer()
matrix_tf = cv.fit_transform(documents_df_post['content'])
tv = TfidfVectorizer()
matrix_tfidf = tv.fit_transform(documents_df_post['content'])

In [ ]:
sparsity_tf = (1 - (matrix_tf.nnz / (matrix_tf.shape[0] * matrix_tf.shape[1]))) * 100

In [ ]:
matrices["post"] = {
    "words": matrix_tf.shape[1],
    "sparsity": sparsity_tf
}


In [ ]:
matrices = pd.DataFrame.from_dict(matrices, orient="index")
display(matrices)

# Przygotowanie katalogów

In [ ]:
create_dir("./wordclouds")
create_dir("./wordclouds/pre")
create_dir("./wordclouds/post")
create_dir("./topic_models")
create_dir("./topic_models/topics")
create_dir("./topic_models/documents")
create_dir("./clustering")
create_dir("./ngrams")

# Worldclouds

In [ ]:
wordcloud = WordCloud(
    background_color="white",
    max_words=5000,
    contour_width=3,
    contour_color="steelblue"
)

In [ ]:
for index, row in documents_df_pre.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/pre/{index.replace(".txt", ".png")}")
    plt.close()

In [ ]:
for index, row in documents_df_post.iterrows():
    wordcloud.generate(row["content"])
    plt.imshow(wordcloud, interpolation="bilinear")
    plt.axis("off")
    plt.title(index.replace(".txt", ""))
    plt.savefig(f"./wordclouds/post/{index.replace(".txt", ".png")}")
    plt.close()